## Question 1

In [ ]:
reactions = [
    'CC(=O)O.OCC>[H+][Cl-]>CC(=O)OCC.O',
    'C=C.[H][H]>[Pd]>CC',
    'c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O'
]

def parse_reaction(rxn):
    p = rxn.split('>')
    if len(p) != 3:
        raise ValueError("Reaction SMILES must contain exactly two '>' separators")
    keys = ['reactants', 'reagents', 'products']
    return {k: [s for s in v.split('.') if s] if v else [] for k, v in zip(keys, p)}

for r in reactions:
    d = parse_reaction(r)
    for k in ['reactants', 'reagents', 'products']:
        print(k, len(d[k]), d[k])
    print()

reactants 2 ['CC(=O)O', 'OCC']
reagents 1 ['[H+][Cl-]']
products 2 ['CC(=O)OCC', 'O']

reactants 2 ['C=C', '[H][H]']
reagents 1 ['[Pd]']
products 1 ['CC']

reactants 2 ['c1ccccc1', 'O=[N+]([O-])O']
reagents 0 []
products 2 ['c1ccccc1[N+](=O)[O-]', 'O']




Reaction 1 is the esterification, and an empty reagent field in the third reaction means that the reaction proceeds without catalyst or solvent.

## Question 2


In [ ]:
import numpy as np

E = np.array([
    [ 2.,  0., -1.,  0.],
    [ 6.,  0.,  0., -2.],
    [ 0.,  2., -2., -1.]
])

U, S, Vt = np.linalg.svd(E)
r = int(np.sum(S > 1e-10))
nullity = E.shape[1] - r

v = Vt[-1]
if v[0] < 0:
    v = -v

x1 = v / v[0]
x_int = np.round(x1 * 2).astype(int)

print("Singular values:", S)
print("Rank:", r, ", Nullity:", nullity)
print("Coefficients scaled to C2H6 = 1:", x1)
print("Smallest whole-number coefficients:", x_int)
print("Product E @ x:", E @ x_int)

Singular values: [6.62561256 3.00720721 1.02857332]
Rank: 3 , Nullity: 1
Coefficients scaled to C2H6 = 1: [1.  3.5 2.  3. ]
Smallest whole-number coefficients: [2 7 4 6]
Product E @ x: [0. 0. 0.]


 A one-dimensional null space means chemically that the stoichiometry of the reaction is uniquely determined up to a single overall scaling constant.

## Question 3

In [ ]:
import numpy as np

A = np.zeros((4, 4))
for i in range(3):
    A[i, i + 1] = A[i + 1, i] = 1.0

deg = np.sum(A, axis=1)
w = np.sort(np.linalg.eigvalsh(A))[::-1]

E_pi = 2 * w[0] + 2 * w[1]
DE = E_pi - 4.0

print("Degrees of atoms:", deg.astype(int))
print("Eigenvalues (descending):", w)
print("Total pi energy (beta term):", E_pi)
print("Delocalisation energy:", DE, "beta")

Degrees of atoms: [1 2 2 1]
Eigenvalues (descending): [ 1.61803399  0.61803399 -0.61803399 -1.61803399]
Total pi energy (beta term): 4.47213595499958
Delocalisation energy: 0.4721359549995796 beta


## Question 4

In [ ]:
import numpy as np

A6 = np.zeros((6, 6))
for i in range(6):
    A6[i, (i - 1) % 6] = A6[i, (i + 1) % 6] = 1.0

At = A6 + np.eye(6)
P = np.linalg.inv(np.diag(np.sum(At, axis=1))) @ At

w = np.sort(np.abs(np.linalg.eigvals(P)))[::-1]
mu = w[1]

rng = np.random.default_rng(1)
H0 = rng.normal(size=(6, 3))

print("Moduli of all 6 eigenvalues:", w)
print("Smoothing rate mu:", mu)
print("Largest deviation from the mean row at step k:")
for k in [0, 1, 2, 4, 8, 16]:
    Hk = np.linalg.matrix_power(P, k) @ H0
    dev = np.max(np.linalg.norm(Hk - np.mean(Hk, axis=0, keepdims=True), axis=1))
    print(f"k = {k:2d}: {dev:.6f}")

Moduli of all 6 eigenvalues: [1.         0.66666667 0.66666667 0.33333333 0.33333333 0.        ]
Smoothing rate mu: 0.6666666666666666
Largest deviation from the mean row at step k:
k =  0: 1.636603
k =  1: 1.018049
k =  2: 0.589886
k =  4: 0.258410
k =  8: 0.051011
k = 16: 0.001990


## Question 5

In [ ]:
import numpy as np

X = np.array([
    [ 78.1, 2.3],
    [ 92.1, 1.7],
    [106.2, 2.8],
    [120.2, 2.0],
    [134.2, 3.1],
    [148.2, 2.4]
])

def run_pca(M, label):
    C = (M.T @ M) / len(M)
    w, V = np.linalg.eigh(C)
    idx = np.argsort(w)[::-1]
    w, V = w[idx], V[:, idx]
    for j in range(V.shape[1]):
        if V[np.argmax(np.abs(V[:, j])), j] < 0:
            V[:, j] = -V[:, j]
    print(f"--- {label} PCA ---")
    print("Covariance matrix:\n", C)
    print("Eigenvalues:", w)
    print("Fraction of variance explained:", w / np.sum(w))
    print("PC1 direction:", V[:, 0])
    print()

Xc = X - np.mean(X, axis=0)
run_pca(Xc, "Centred")

Xs = Xc / np.std(X, axis=0, ddof=0)
run_pca(Xs, "Standardised")

--- Centred PCA ---
Covariance matrix:
 [[5.73535556e+02 4.56277778e+00]
 [4.56277778e+00 2.18055556e-01]]
Eigenvalues: [5.73571866e+02 1.81744746e-01]
Fraction of variance explained: [9.99683236e-01 3.16764448e-04]
PC1 direction: [0.99996834 0.0079578 ]

--- Standardised PCA ---
Covariance matrix:
 [[1.         0.40800508]
 [0.40800508 1.        ]]
Eigenvalues: [1.40800508 0.59199492]
Fraction of variance explained: [0.70400254 0.29599746]
PC1 direction: [0.70710678 0.70710678]

